# 17.11 Activation 也量化會改變什麼？


只量化權重時，每個輸入的中間特徵仍是浮點數。若也把這些特徵放到低位元格子，是否只是再省一點空間？它還會增加另一條誤差來源，而且特徵每次輸入都不同。這裡activation指各層的輸入或中間特徵，不是單指GELU等啟動函數。前置是[只量化權重](https://birdhackor.github.io/tiny-perceptron-vlm/17.7.html)和[輸入如何放大或抵消權重誤差](https://birdhackor.github.io/tiny-perceptron-vlm/17.3.html)。

先用同一刻度0.5，隔離兩種誤差。權重w兩列 `[0.7,0.2]`、`[-0.7,1.2]`，兩筆輸入x也取這兩列。權重還原是 `[0.5,0]`與 `[-0.5,1]`；只改w時保留原x。若連x也改成上述還原值，兩邊的差會一起進入點積。每項輸出誤差未必都變大，有的正好抵消，但多了一個需要評估的改動。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch

x = torch.tensor([[0.7, 0.2], [-0.7, 1.2]])
w = torch.tensor([[0.7, 0.2], [-0.7, 1.2]])
scale = 0.5


def simulate_quantization(t):
    return (t / scale).round() * scale


qx, qw = simulate_quantization(x), simulate_quantization(w)
reference = x @ w.T
weight_only = x @ qw.T
both = qx @ qw.T
print("原輸出", reference.round(decimals=4).tolist())
for name, output in (("只改權重", weight_only), ("權重加特徵", both)):
    print(name, output.round(decimals=4).tolist(), "MAE", round((reference - output).abs().mean().item(), 4))

函數先量化再反量化，返回的仍是浮點tensor，用來模擬格子誤差，沒有打包，也沒有使用可讀取低位元編碼的專用運算程式（kernel）。scale刻意固定，真實權重與activation通常可能使用不同刻度；當前不談如何估計，先把兩邊放進相同數值規則。

以下數字取近似值；實際列印可能顯示浮點小數的長尾，讀法見[17.3](https://birdhackor.github.io/tiny-perceptron-vlm/17.3.html)。原輸出約為 `[[0.53,-0.25],[-0.25,1.93]]`。只改權重得到約 `[[0.35,-0.15],[-0.35,1.55]]`，MAE0.19；兩邊都改得到 `[[0.25,-0.25],[-0.25,1.25]]`，MAE0.24。觀察非對角兩項：權重單獨量化時有誤差，加入特徵量化反而剛好回到原值-0.25。這不是整體更好，而是誤差方向偶然抵消；最後1.93變1.25則更差。

權重固定，activation的範圍卻隨句子長度、內容或模態改變。只在一個輸入上選得合適的刻度，其他輸入可能碰到截斷或細節消失；多層誤差還會影響後續層的特徵分布。因此要分別記錄權重與特徵的位元數、分組方式、可表示的最小與最大數值，以及超出範圍時是否壓回端點等處理規則，再用獨立資料做完整任務檢查，不能只用單層平均誤差排名。

練習只把x第一列改成 `[1,0]`，w保持不變。先預測這列已在0.5格子上，所以「只改權重」與「兩邊都改」的第一列輸出相同，都是 `[0.5,-0.5]`；再執行核對。這個練習能直接檢驗：多一個量化位置不表示每個輸入都一定新增誤差，關鍵在它落在哪些格子。
